# Closing Auction Uncross Price Distribution V1

Real closing auction orderbook feeds are proprietary. This notebook therefore builds a prototype using a synthetic auction simulator that mimics the structure of indicative uncross price, indicative uncross volume, imbalance, and final uncross price data.

The goal is to demonstrate the modeling workflow: define the target as a distribution, engineer auction-state features, train quantile regressors, evaluate prediction interval coverage, and show how real-time inference would update as new auction messages arrive.

## 1. Imports and Configuration

In [ ]:
from __future__ import annotations

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_pinball_loss
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

RAW_DIR = ROOT / 'data' / 'raw'
PROCESSED_DIR = ROOT / 'data' / 'processed'
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
N_AUCTIONS = 2500
SNAPSHOTS_PER_AUCTION = 20
QUANTILES = [0.10, 0.25, 0.50, 0.75, 0.90]
rng = np.random.default_rng(RANDOM_SEED)

## 2. Synthetic Auction Data

Each auction contains a sequence of snapshots. The model sees the state at a snapshot and predicts the final uncross price move relative to the current indicative uncross price.

In [ ]:
def simulate_auction_data(n_auctions: int, snapshots_per_auction: int) -> pd.DataFrame:
    rows = []
    tickers = [f'STK{i:03d}' for i in range(60)]

    for auction_id in range(n_auctions):
        ticker = rng.choice(tickers)
        base_price = rng.lognormal(mean=4.2, sigma=0.35)
        daily_volatility = rng.uniform(0.008, 0.045)
        typical_auction_volume = rng.lognormal(mean=12.0, sigma=0.7)
        index_rebalance = rng.random() < 0.08
        latent_pressure = rng.normal(0, 1.0) + (rng.choice([-1, 1]) * rng.uniform(0.5, 1.5) if index_rebalance else 0)
        last_continuous_price = base_price * (1 + rng.normal(0, daily_volatility / 2))
        iup = last_continuous_price
        iup_path = []

        for snapshot in range(snapshots_per_auction):
            seconds_to_close = (snapshots_per_auction - snapshot) * 15
            time_fraction = 1 - seconds_to_close / (snapshots_per_auction * 15)
            volume_scale = 0.2 + 1.8 * time_fraction + rng.normal(0, 0.08)
            auction_volume = max(1.0, typical_auction_volume * volume_scale)
            imbalance = np.tanh(latent_pressure + rng.normal(0, 0.8) * (1 - time_fraction))
            iup_change = 0.0008 * imbalance + rng.normal(0, daily_volatility / 22)
            iup = max(0.1, iup * (1 + iup_change))
            iup_path.append(iup)

            rows.append({
                'auction_id': auction_id,
                'ticker': ticker,
                'snapshot': snapshot,
                'seconds_to_close': seconds_to_close,
                'last_continuous_price': last_continuous_price,
                'current_iup': iup,
                'auction_volume': auction_volume,
                'typical_auction_volume': typical_auction_volume,
                'imbalance': imbalance,
                'daily_volatility': daily_volatility,
                'index_rebalance': int(index_rebalance),
            })

        final_noise = rng.normal(0, daily_volatility / 18)
        final_move = 0.0015 * np.tanh(latent_pressure) + 0.35 * ((iup_path[-1] / iup_path[max(0, len(iup_path) - 6)]) - 1) + final_noise
        final_uncross_price = iup_path[-1] * (1 + final_move)
        for row in rows[-snapshots_per_auction:]:
            row['final_uncross_price'] = final_uncross_price

    return pd.DataFrame(rows)


auction = simulate_auction_data(N_AUCTIONS, SNAPSHOTS_PER_AUCTION)
auction.to_csv(RAW_DIR / 'synthetic_closing_auction_messages.csv', index=False)
print(auction.shape)
auction.head()

## 3. Feature Engineering and Target

The target is the final uncross return relative to the current indicative uncross price. A positive target means the final price cleared above the current IUP.

In [ ]:
df = auction.sort_values(['auction_id', 'snapshot']).copy()
df['iup_return_from_last_price'] = df['current_iup'] / df['last_continuous_price'] - 1
df['auction_volume_ratio'] = df['auction_volume'] / df['typical_auction_volume']
df['round_number_distance'] = (df['current_iup'] - df['current_iup'].round()) / df['current_iup']
df['iup_change_1'] = df.groupby('auction_id')['current_iup'].pct_change().fillna(0)
df['iup_trend_5'] = df.groupby('auction_id')['current_iup'].pct_change(5).fillna(0)
df['imbalance_change_1'] = df.groupby('auction_id')['imbalance'].diff().fillna(0)
df['abs_imbalance'] = df['imbalance'].abs()
df['target_final_move'] = df['final_uncross_price'] / df['current_iup'] - 1

feature_cols = [
    'seconds_to_close', 'iup_return_from_last_price', 'auction_volume_ratio', 'imbalance',
    'abs_imbalance', 'imbalance_change_1', 'iup_change_1', 'iup_trend_5',
    'round_number_distance', 'daily_volatility', 'index_rebalance'
]

model_df = df.dropna(subset=feature_cols + ['target_final_move']).reset_index(drop=True)
model_df.to_csv(PROCESSED_DIR / 'closing_auction_features_v1.csv', index=False)
print(model_df.shape)
model_df[feature_cols + ['target_final_move']].head()

## 4. Train Quantile Models

Quantile regression estimates several points of the conditional distribution instead of only predicting the mean.

In [ ]:
unique_auctions = model_df['auction_id'].drop_duplicates()
train_ids, test_ids = train_test_split(unique_auctions, test_size=0.25, random_state=RANDOM_SEED)
train = model_df[model_df['auction_id'].isin(train_ids)].copy()
test = model_df[model_df['auction_id'].isin(test_ids)].copy()

X_train, y_train = train[feature_cols], train['target_final_move']
X_test, y_test = test[feature_cols], test['target_final_move']

models = {}
preds = test[['auction_id', 'ticker', 'snapshot', 'seconds_to_close', 'current_iup', 'final_uncross_price', 'target_final_move']].copy()

for q in QUANTILES:
    model = GradientBoostingRegressor(
        loss='quantile', alpha=q, n_estimators=250, max_depth=3, learning_rate=0.04, random_state=RANDOM_SEED
    )
    model.fit(X_train, y_train)
    models[q] = model
    preds[f'q{int(q * 100):02d}'] = model.predict(X_test)

median_model = models[0.50]
median_pred = preds['q50']
print('Median MAE:', mean_absolute_error(y_test, median_pred))
preds.head()

## 5. Distribution Evaluation

In [ ]:
pinball = pd.DataFrame({
    'quantile': QUANTILES,
    'pinball_loss': [mean_pinball_loss(y_test, preds[f'q{int(q * 100):02d}'], alpha=q) for q in QUANTILES]
})
display(pinball)

coverage_80 = ((y_test >= preds['q10']) & (y_test <= preds['q90'])).mean()
coverage_50 = ((y_test >= preds['q25']) & (y_test <= preds['q75'])).mean()
print(f'80% interval coverage: {coverage_80:.3f}')
print(f'50% interval coverage: {coverage_50:.3f}')

plt.figure(figsize=(7, 4))
sns.scatterplot(x=preds['q50'], y=y_test, alpha=0.25, s=12)
plt.axline((0, 0), slope=1, color='black', linestyle='--')
plt.title('Predicted Median vs Realized Final Uncross Move')
plt.xlabel('Predicted median move')
plt.ylabel('Realized move')
plt.show()

## 6. Real-Time Inference Demo

In production, each incoming auction message would update the feature vector and refresh the predicted distribution. This block shows that behavior on one simulated auction.

In [ ]:
sample_id = preds['auction_id'].iloc[0]
stream = model_df[model_df['auction_id'] == sample_id].sort_values('snapshot').copy()
stream_preds = stream[['snapshot', 'seconds_to_close', 'target_final_move']].copy()
for q, model in models.items():
    stream_preds[f'q{int(q * 100):02d}'] = model.predict(stream[feature_cols])

display(stream_preds.tail())

plt.figure(figsize=(9, 5))
plt.plot(stream_preds['seconds_to_close'], stream_preds['q50'], label='median forecast')
plt.fill_between(stream_preds['seconds_to_close'], stream_preds['q10'], stream_preds['q90'], alpha=0.25, label='10-90% interval')
plt.axhline(stream_preds['target_final_move'].iloc[-1], color='black', linestyle='--', label='realized final move')
plt.gca().invert_xaxis()
plt.title('Real-Time Distribution Forecast During One Auction')
plt.xlabel('Seconds to close')
plt.ylabel('Final move from current IUP')
plt.legend()
plt.show()

## 7. What Would Change With Real Data

- Replace the simulator with LSE, NASDAQ, NYSE, LOBSTER, TAQ, or vendor auction messages.
- Preserve exact exchange timestamps and message ordering.
- Add symbol-level historical auction volume baselines and index rebalance calendars.
- Evaluate calibration by stock, time-to-close bucket, volatility regime, and auction imbalance bucket.
- Add conformal prediction intervals after the quantile models are stable.